# Global Solution - Smart Water & Energy Station (Versão Interativa no Colab)
**Disciplinas:** Advanced Programming & Mobile Dev | Técnicas Avançadas de Programação

**Integrantes:**
- Rafael Lucca Silva Freitas Bazan - RM 561920
- Gabriel Assis - RM 562280
- Fernando Castanha - RM 562226

---

### Célula 1: Preparar o ambiente no Google Colab (Java 17, Maven e Estrutura de Pastas)

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq openjdk-17-jdk maven zip

mkdir -p backend-estacao/src/main/java/br/com/fiap/estacao/model
mkdir -p backend-estacao/src/main/java/br/com/fiap/estacao/repository
mkdir -p backend-estacao/src/main/java/br/com/fiap/estacao/controller
mkdir -p backend-estacao/src/main/resources

mkdir -p mobile-estacao/src/components
mkdir -p mobile-estacao/src/screens
mkdir -p mobile-estacao/src/services

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


### Célula 2: Configurar o Backend Spring Boot e o Banco H2 em Modo Arquivo (`jdbc:h2:file:./data/estacao`)

In [ ]:
%%writefile backend-estacao/pom.xml
<?xml version="1.0" encoding="UTF-8"?>
<project xmlns="http://maven.apache.org/POM/4.0.0"
         xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance"
         xsi:schemaLocation="http://maven.apache.org/POM/4.0.0 https://maven.apache.org/xsd/maven-4.0.0.xsd">
    <modelVersion>4.0.0</modelVersion>
    <parent>
        <groupId>org.springframework.boot</groupId>
        <artifactId>spring-boot-starter-parent</artifactId>
        <version>3.2.5</version>
        <relativePath/>
    </parent>
    <groupId>br.com.fiap</groupId>
    <artifactId>estacao</artifactId>
    <version>1.0.0</version>
    <name>Smart Water and Energy Station Backend</name>

    <properties>
        <java.version>17</java.version>
    </properties>

    <dependencies>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-web</artifactId>
        </dependency>
        <dependency>
            <groupId>org.springframework.boot</groupId>
            <artifactId>spring-boot-starter-data-jpa</artifactId>
        </dependency>
        <dependency>
            <groupId>com.h2database</groupId>
            <artifactId>h2</artifactId>
            <scope>runtime</scope>
        </dependency>
    </dependencies>

    <build>
        <plugins>
            <plugin>
                <groupId>org.springframework.boot</groupId>
                <artifactId>spring-boot-maven-plugin</artifactId>
            </plugin>
        </plugins>
    </build>
</project>

Overwriting backend-estacao/pom.xml


In [ ]:
%%writefile backend-estacao/src/main/resources/application.properties
server.port=8080
spring.datasource.url=jdbc:h2:file:./data/estacao
spring.datasource.driverClassName=org.h2.Driver
spring.datasource.username=sa
spring.datasource.password=
spring.jpa.database-platform=org.hibernate.dialect.H2Dialect
spring.jpa.hibernate.ddl-auto=update
spring.h2.console.enabled=true
spring.h2.console.path=/h2-console

Overwriting backend-estacao/src/main/resources/application.properties


### Célula 3: Código-Fonte do Backend (Entidades, Repositórios e Controllers)

In [ ]:
%%writefile backend-estacao/src/main/java/br/com/fiap/estacao/EstacaoApplication.java
package br.com.fiap.estacao;

import org.springframework.boot.SpringApplication;
import org.springframework.boot.autoconfigure.SpringBootApplication;

@SpringBootApplication
public class EstacaoApplication {
    public static void main(String[] args) {
        SpringApplication.run(EstacaoApplication.class, args);
    }
}

Overwriting backend-estacao/src/main/java/br/com/fiap/estacao/EstacaoApplication.java


In [ ]:
%%writefile backend-estacao/src/main/java/br/com/fiap/estacao/model/Leitura.java
package br.com.fiap.estacao.model;

import jakarta.persistence.*;
import java.time.LocalDateTime;

@Entity
public class Leitura {
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;
    private Double nivelAgua;
    private Double vazao;
    private Double consumoEletrico;
    private LocalDateTime dataHora = LocalDateTime.now();

    public Long getId() { return id; }
    public void setId(Long id) { this.id = id; }
    public Double getNivelAgua() { return nivelAgua; }
    public void setNivelAgua(Double nivelAgua) { this.nivelAgua = nivelAgua; }
    public Double getVazao() { return vazao; }
    public void setVazao(Double vazao) { this.vazao = vazao; }
    public Double getConsumoEletrico() { return consumoEletrico; }
    public void setConsumoEletrico(Double consumoEletrico) { this.consumoEletrico = consumoEletrico; }
    public LocalDateTime getDataHora() { return dataHora; }
    public void setDataHora(LocalDateTime dataHora) { this.dataHora = dataHora; }
}

Overwriting backend-estacao/src/main/java/br/com/fiap/estacao/model/Leitura.java


In [ ]:
%%writefile backend-estacao/src/main/java/br/com/fiap/estacao/model/Atuador.java
package br.com.fiap.estacao.model;

import jakarta.persistence.*;
import java.time.LocalDateTime;

@Entity
public class Atuador {
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;
    private String nome;
    private String estado;
    private LocalDateTime ultimaAtualizacao = LocalDateTime.now();

    public Long getId() { return id; }
    public void setId(Long id) { this.id = id; }
    public String getNome() { return nome; }
    public void setNome(String nome) { this.nome = nome; }
    public String getEstado() { return estado; }
    public void setEstado(String estado) { this.estado = estado; }
    public LocalDateTime getUltimaAtualizacao() { return ultimaAtualizacao; }
    public void setUltimaAtualizacao(LocalDateTime ultimaAtualizacao) { this.ultimaAtualizacao = ultimaAtualizacao; }
}

Overwriting backend-estacao/src/main/java/br/com/fiap/estacao/model/Atuador.java


In [ ]:
%%writefile backend-estacao/src/main/java/br/com/fiap/estacao/model/Falha.java
package br.com.fiap.estacao.model;

import jakarta.persistence.*;
import java.time.LocalDateTime;

@Entity
public class Falha {
    @Id
    @GeneratedValue(strategy = GenerationType.IDENTITY)
    private Long id;
    private String descricao;
    private String severidade;
    private LocalDateTime dataHora = LocalDateTime.now();

    public Falha() {}

    public Falha(String descricao, String severidade) {
        this.descricao = descricao;
        this.severidade = severidade;
        this.dataHora = LocalDateTime.now();
    }

    public Long getId() { return id; }
    public void setId(Long id) { this.id = id; }
    public String getDescricao() { return descricao; }
    public void setDescricao(String descricao) { this.descricao = descricao; }
    public String getSeveridade() { return severidade; }
    public void setSeveridade(String severidade) { this.severidade = severidade; }
    public LocalDateTime getDataHora() { return dataHora; }
    public void setDataHora(LocalDateTime dataHora) { this.dataHora = dataHora; }
}

Overwriting backend-estacao/src/main/java/br/com/fiap/estacao/model/Falha.java


In [ ]:
%%writefile backend-estacao/src/main/java/br/com/fiap/estacao/repository/LeituraRepository.java
package br.com.fiap.estacao.repository;

import br.com.fiap.estacao.model.Leitura;
import org.springframework.data.jpa.repository.JpaRepository;

public interface LeituraRepository extends JpaRepository<Leitura, Long> {}

Overwriting backend-estacao/src/main/java/br/com/fiap/estacao/repository/LeituraRepository.java


In [ ]:
%%writefile backend-estacao/src/main/java/br/com/fiap/estacao/repository/AtuadorRepository.java
package br.com.fiap.estacao.repository;

import br.com.fiap.estacao.model.Atuador;
import org.springframework.data.jpa.repository.JpaRepository;

public interface AtuadorRepository extends JpaRepository<Atuador, Long> {}

Overwriting backend-estacao/src/main/java/br/com/fiap/estacao/repository/AtuadorRepository.java


In [ ]:
%%writefile backend-estacao/src/main/java/br/com/fiap/estacao/repository/FalhaRepository.java
package br.com.fiap.estacao.repository;

import br.com.fiap.estacao.model.Falha;
import org.springframework.data.jpa.repository.JpaRepository;

public interface FalhaRepository extends JpaRepository<Falha, Long> {}

Overwriting backend-estacao/src/main/java/br/com/fiap/estacao/repository/FalhaRepository.java


In [ ]:
%%writefile backend-estacao/src/main/java/br/com/fiap/estacao/controller/EstacaoController.java
package br.com.fiap.estacao.controller;

import br.com.fiap.estacao.model.Atuador;
import br.com.fiap.estacao.model.Falha;
import br.com.fiap.estacao.model.Leitura;
import br.com.fiap.estacao.repository.AtuadorRepository;
import br.com.fiap.estacao.repository.FalhaRepository;
import br.com.fiap.estacao.repository.LeituraRepository;
import org.springframework.web.bind.annotation.*;

import java.time.LocalDateTime;
import java.util.List;

@RestController
@RequestMapping("/api")
@CrossOrigin(origins = "*")
public class EstacaoController {

    private final LeituraRepository leituraRepository;
    private final AtuadorRepository atuadorRepository;
    private final FalhaRepository falhaRepository;

    public EstacaoController(LeituraRepository leituraRepository,
                             AtuadorRepository atuadorRepository,
                             FalhaRepository falhaRepository) {
        this.leituraRepository = leituraRepository;
        this.atuadorRepository = atuadorRepository;
        this.falhaRepository = falhaRepository;
    }

    @PostMapping("/leituras")
    public Leitura registrarLeitura(@RequestBody Leitura leitura) {
        leitura.setDataHora(LocalDateTime.now());
        Leitura salva = leituraRepository.save(leitura);

        if (leitura.getNivelAgua() != null && (leitura.getNivelAgua() < 20.0 || leitura.getNivelAgua() > 90.0)) {
            falhaRepository.save(new Falha("Nível de água fora do limite seguro: " + leitura.getNivelAgua() + "%", "ALTA"));
        }
        if (leitura.getConsumoEletrico() != null && leitura.getConsumoEletrico() > 500.0) {
            falhaRepository.save(new Falha("Sobrecarga de consumo elétrico: " + leitura.getConsumoEletrico() + " kWh", "MEDIA"));
        }
        return salva;
    }

    @GetMapping("/leituras")
    public List<Leitura> listarLeituras() {
        return leituraRepository.findAll();
    }

    @PostMapping("/atuadores")
    public Atuador acionarAtuador(@RequestBody Atuador atuador) {
        atuador.setUltimaAtualizacao(LocalDateTime.now());
        return atuadorRepository.save(atuador);
    }

    @GetMapping("/atuadores")
    public List<Atuador> listarAtuadores() {
        return atuadorRepository.findAll();
    }

    @GetMapping("/falhas")
    public List<Falha> listarFalhas() {
        return falhaRepository.findAll();
    }

    @PostMapping("/falhas")
    public Falha registrarFalha(@RequestBody Falha falha) {
        falha.setDataHora(LocalDateTime.now());
        return falhaRepository.save(falha);
    }
}

Overwriting backend-estacao/src/main/java/br/com/fiap/estacao/controller/EstacaoController.java


### Célula 4: README do Backend

In [ ]:
%%writefile backend-estacao/README.md
# Smart Water & Energy Station - Backend (Spring Boot + H2)

## Integrantes
- Rafael Lucca Silva Freitas Bazan - RM 561920
- Gabriel Assis - RM 562280
- Fernando Castanha - RM 562226

## Descrição do Projeto
API REST desenvolvida em Java com Spring Boot e persistência em banco de dados H2 em modo arquivo (`jdbc:h2:file:./data/estacao`) para operação da Smart Water & Energy Station.

## Como Executar
1. Certifique-se de possuir o Java 17+ e o Maven instalados.
2. No diretório `backend-estacao`, execute:
   ```bash
   mvn spring-boot:run
   ```
3. A API estará disponível em `http://localhost:8080/api`.
4. Os dados ficam salvos no arquivo `./data/estacao.mv.db` e permanecem gravados após reiniciar a aplicação.

## Endpoints Documentados

### 1. Registrar Leitura
- **Método:** `POST`
- **URL:** `/api/leituras`
- **Body (JSON):**
  ```json
  {
    "nivelAgua": 15.0,
    "vazao": 42.5,
    "consumoEletrico": 520.0
  }
  ```

### 2. Listar Leituras
- **Método:** `GET`
- **URL:** `/api/leituras`

### 3. Acionar Atuador
- **Método:** `POST`
- **URL:** `/api/atuadores`
- **Body (JSON):**
  ```json
  {
    "nome": "Bomba Principal",
    "estado": "LIGADO"
  }
  ```

### 4. Listar Atuadores
- **Método:** `GET`
- **URL:** `/api/atuadores`

### 5. Listar Falhas
- **Método:** `GET`
- **URL:** `/api/falhas`

### 6. Registrar Falha Manual
- **Método:** `POST`
- **URL:** `/api/falhas`
- **Body (JSON):**
  ```json
  {
    "descricao": "Falha no sensor de vazão",
    "severidade": "ALTA"
  }
  ```

Overwriting backend-estacao/README.md


### Célula 5: Código-Fonte do Aplicativo Mobile (React Native)

In [ ]:
%%writefile mobile-estacao/package.json
{
  "name": "mobile-estacao",
  "version": "1.0.0",
  "main": "node_modules/expo/AppEntry.js",
  "scripts": {
    "start": "expo start",
    "android": "expo start --android",
    "ios": "expo start --ios",
    "web": "expo start --web"
  },
  "dependencies": {
    "expo": "~51.0.0",
    "react": "18.2.0",
    "react-native": "0.74.1",
    "@react-navigation/native": "^6.1.17",
    "@react-navigation/bottom-tabs": "^6.5.20",
    "react-native-screens": "~3.31.1",
    "react-native-safe-area-context": "4.10.1"
  }
}

Overwriting mobile-estacao/package.json


In [ ]:
%%writefile mobile-estacao/src/services/api.js
const API_URL = 'http://localhost:8080/api';

export async function buscarDadosEstacao() {
  const [resLeituras, resFalhas, resAtuadores] = await Promise.all([
    fetch(`${API_URL}/leituras`),
    fetch(`${API_URL}/falhas`),
    fetch(`${API_URL}/atuadores`)
  ]);

  if (!resLeituras.ok || !resFalhas.ok || !resAtuadores.ok) {
    throw new Error('API fora do ar');
  }

  return {
    leituras: await resLeituras.json(),
    falhas: await resFalhas.json(),
    atuadores: await resAtuadores.json()
  };
}

export async function enviarLeitura(leitura) {
  const response = await fetch(`${API_URL}/leituras`, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify(leitura)
  });
  if (!response.ok) {
    throw new Error('API fora do ar');
  }
  return response.json();
}

export async function acionarAtuadorApi(atuador) {
  const response = await fetch(`${API_URL}/atuadores`, {
    method: 'POST',
    headers: { 'Content-Type': 'application/json' },
    body: JSON.stringify(atuador)
  });
  if (!response.ok) {
    throw new Error('API fora do ar');
  }
  return response.json();
}

Overwriting mobile-estacao/src/services/api.js


In [ ]:
%%writefile mobile-estacao/src/components/IndicadorCard.js
import React from 'react';
import { View, Text, StyleSheet } from 'react-native';

export default function IndicadorCard({ titulo, valor, unidade }) {
  return (
    <View style={styles.card}>
      <Text style={styles.titulo}>{titulo}</Text>
      <Text style={styles.valor}>
        {valor !== null && valor !== undefined ? `${valor} ${unidade}` : '--'}
      </Text>
    </View>
  );
}

const styles = StyleSheet.create({
  card: {
    backgroundColor: '#ffffff',
    padding: 16,
    borderRadius: 8,
    marginBottom: 12,
    borderWidth: 1,
    borderColor: '#dcdcdc'
  },
  titulo: {
    fontSize: 14,
    color: '#555555'
  },
  valor: {
    fontSize: 22,
    fontWeight: 'bold',
    color: '#111111',
    marginTop: 4
  }
});

Overwriting mobile-estacao/src/components/IndicadorCard.js


In [ ]:
%%writefile mobile-estacao/src/components/BannerErro.js
import React from 'react';
import { View, Text, StyleSheet } from 'react-native';

export default function BannerErro({ visivel, mensagem }) {
  if (!visivel) return null;
  return (
    <View style={styles.banner}>
      <Text style={styles.texto}>{mensagem || 'API fora do ar'}</Text>
    </View>
  );
}

const styles = StyleSheet.create({
  banner: {
    backgroundColor: '#d9534f',
    padding: 12,
    borderRadius: 6,
    marginBottom: 12
  },
  texto: {
    color: '#ffffff',
    fontWeight: 'bold',
    textAlign: 'center'
  }
});

Overwriting mobile-estacao/src/components/BannerErro.js


In [ ]:
%%writefile mobile-estacao/src/screens/PainelScreen.js
import React, { useEffect, useState } from 'react';
import { View, Text, Button, StyleSheet, ScrollView } from 'react-native';
import IndicadorCard from '../components/IndicadorCard';
import BannerErro from '../components/BannerErro';
import { buscarDadosEstacao, enviarLeitura, acionarAtuadorApi } from '../services/api';

export default function PainelScreen() {
  const [ultimaLeitura, setUltimaLeitura] = useState(null);
  const [atuadores, setAtuadores] = useState([]);
  const [erroApi, setErroApi] = useState(false);

  const carregarPainel = async () => {
    try {
      const dados = await buscarDadosEstacao();
      if (dados.leituras.length > 0) {
        setUltimaLeitura(dados.leituras[dados.leituras.length - 1]);
      }
      setAtuadores(dados.atuadores);
      setErroApi(false);
    } catch (e) {
      setErroApi(true);
    }
  };

  const simularLeitura = async () => {
    try {
      await enviarLeitura({
        nivelAgua: Number((Math.random() * 100).toFixed(1)),
        vazao: Number((Math.random() * 60).toFixed(1)),
        consumoEletrico: Number((Math.random() * 600).toFixed(1))
      });
      await carregarPainel();
    } catch (e) {
      setErroApi(true);
    }
  };

  const alternarBomba = async () => {
    try {
      const ultimoEstado = atuadores.length > 0 ? atuadores[atuadores.length - 1].estado : 'DESLIGADO';
      const novoEstado = ultimoEstado === 'LIGADO' ? 'DESLIGADO' : 'LIGADO';
      await acionarAtuadorApi({ nome: 'Bomba Principal', estado: novoEstado });
      await carregarPainel();
    } catch (e) {
      setErroApi(true);
    }
  };

  useEffect(() => {
    carregarPainel();
  }, []);

  return (
    <ScrollView contentContainerStyle={styles.container}>
      <BannerErro visivel={erroApi} mensagem="API fora do ar" />

      <IndicadorCard
        titulo="Nível da Smart Water & Energy Station"
        valor={ultimaLeitura?.nivelAgua}
        unidade="%"
      />
      <IndicadorCard
        titulo="Vazão"
        valor={ultimaLeitura?.vazao}
        unidade="L/s"
      />
      <IndicadorCard
        titulo="Consumo Elétrico Simulado"
        valor={ultimaLeitura?.consumoEletrico}
        unidade="kWh"
      />

      <View style={styles.blocoBotoes}>
        <Button title="Atualizar Dados" onPress={carregarPainel} />
      </View>
      <View style={styles.blocoBotoes}>
        <Button title="Registrar Leitura Simulada" onPress={simularLeitura} />
      </View>
      <View style={styles.blocoBotoes}>
        <Button title="Acionar Atuador (Bomba Principal)" onPress={alternarBomba} />
      </View>

      <Text style={styles.subtitulo}>Estado dos Atuadores:</Text>
      {atuadores.map((at) => (
        <Text key={at.id} style={styles.itemAtuador}>
          {at.nome}: {at.estado}
        </Text>
      ))}
    </ScrollView>
  );
}

const styles = StyleSheet.create({
  container: {
    padding: 16,
    backgroundColor: '#f5f5f5',
    flexGrow: 1
  },
  blocoBotoes: {
    marginBottom: 10
  },
  subtitulo: {
    fontSize: 16,
    fontWeight: 'bold',
    marginTop: 12,
    marginBottom: 6
  },
  itemAtuador: {
    fontSize: 14,
    marginBottom: 4
  }
});

Overwriting mobile-estacao/src/screens/PainelScreen.js


In [ ]:
%%writefile mobile-estacao/src/screens/FalhasScreen.js
import React, { useEffect, useState } from 'react';
import { View, Text, FlatList, Button, StyleSheet } from 'react-native';
import BannerErro from '../components/BannerErro';
import { buscarDadosEstacao } from '../services/api';

export default function FalhasScreen() {
  const [falhas, setFalhas] = useState([]);
  const [erroApi, setErroApi] = useState(false);

  const carregarFalhas = async () => {
    try {
      const dados = await buscarDadosEstacao();
      setFalhas(dados.falhas);
      setErroApi(false);
    } catch (e) {
      setErroApi(true);
    }
  };

  useEffect(() => {
    carregarFalhas();
  }, []);

  return (
    <View style={styles.container}>
      <BannerErro visivel={erroApi} mensagem="API fora do ar" />
      <Button title="Atualizar Alertas de Falha" onPress={carregarFalhas} />
      <FlatList
        data={falhas}
        keyExtractor={(item) => String(item.id)}
        renderItem={({ item }) => (
          <View style={styles.cardFalha}>
            <Text style={styles.severidade}>Severidade: {item.severidade}</Text>
            <Text style={styles.descricao}>{item.descricao}</Text>
            <Text style={styles.data}>{item.dataHora}</Text>
          </View>
        )}
        ListEmptyComponent={<Text style={styles.vazio}>Nenhuma falha registrada.</Text>}
      />
    </View>
  );
}

const styles = StyleSheet.create({
  container: {
    flex: 1,
    padding: 16,
    backgroundColor: '#f5f5f5'
  },
  cardFalha: {
    backgroundColor: '#fff3cd',
    padding: 12,
    borderRadius: 6,
    marginTop: 10,
    borderWidth: 1,
    borderColor: '#ffeeba'
  },
  severidade: {
    fontWeight: 'bold',
    color: '#856404'
  },
  descricao: {
    marginTop: 4,
    color: '#333333'
  },
  data: {
    marginTop: 4,
    fontSize: 12,
    color: '#666666'
  },
  vazio: {
    marginTop: 16,
    textAlign: 'center',
    color: '#666666'
  }
});

Overwriting mobile-estacao/src/screens/FalhasScreen.js


In [ ]:
%%writefile mobile-estacao/App.js
import React from 'react';
import { NavigationContainer } from '@react-navigation/native';
import { createBottomTabNavigator } from '@react-navigation/bottom-tabs';
import PainelScreen from './src/screens/PainelScreen';
import FalhasScreen from './src/screens/FalhasScreen';

const Tab = createBottomTabNavigator();

export default function App() {
  return (
    <NavigationContainer>
      <Tab.Navigator>
        <Tab.Screen name="Painel de Operação" component={PainelScreen} />
        <Tab.Screen name="Alertas de Falha" component={FalhasScreen} />
      </Tab.Navigator>
    </NavigationContainer>
  );
}

Overwriting mobile-estacao/App.js


### Célula 6: README do Aplicativo Mobile

In [ ]:
%%writefile mobile-estacao/README.md
# Smart Water & Energy Station - Aplicativo Mobile (React Native)

## Integrantes
- Rafael Lucca Silva Freitas Bazan - RM 561920
- Gabriel Assis - RM 562280
- Fernando Castanha - RM 562226

## Descrição do Projeto
Aplicativo mobile desenvolvido em React Native (Expo) que funciona como o painel de operação da Smart Water & Energy Station. O aplicativo apresenta:
- Níveis da Smart Water & Energy Station
- Vazão
- Consumo elétrico simulado
- Alertas de falha
- Tratamento de indisponibilidade da API exibindo o aviso "API fora do ar" sem travar a aplicação.

## Como Executar
1. Instale as dependências no diretório `mobile-estacao`:
   ```bash
   npm install
   ```
2. Inicie o aplicativo:
   ```bash
   npx expo start
   ```

Overwriting mobile-estacao/README.md


### Célula 7: Iniciar (ou Reiniciar) o Backend Spring Boot + H2 em 2º Plano no Colab

In [ ]:
import subprocess, time, urllib.request

processo_backend = subprocess.Popen(
    ["mvn", "spring-boot:run"],
    cwd="/content/backend-estacao",
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("A iniciar o Backend Spring Boot + H2...")
for _ in range(45):
    try:
        urllib.request.urlopen("http://localhost:8080/api/leituras")
        print("Backend ONLINE na porta 8080! Dados persistidos em ./data/estacao")
        break
    except Exception:
        time.sleep(2)

A iniciar o Backend Spring Boot + H2...


### Célula 8: Painel Mobile Interativo no Colab (Consumo da API em Tempo Real)

In [ ]:
from google.colab import output
from IPython.display import HTML, display

url_api = output.eval_js("google.colab.kernel.proxyPort(8080)")

html_app = f"""
<div style="width:360px; font-family:sans-serif; background:#f5f5f5; border:2px solid #333; border-radius:20px; padding:16px; box-sizing:border-box;">
  <h3 style="margin-top:0; text-align:center; color:#111;">Smart Water & Energy</h3>

  <div style="display:flex; gap:8px; margin-bottom:12px;">
    <button onclick="trocarAba('painel')" style="flex:1; padding:8px; cursor:pointer; font-weight:bold;">Painel de Operação</button>
    <button onclick="trocarAba('falhas')" style="flex:1; padding:8px; cursor:pointer; font-weight:bold;">Alertas de Falha</button>
  </div>

  <div id="banner-erro" style="display:none; background:#d9534f; color:#fff; padding:10px; border-radius:6px; text-align:center; font-weight:bold; margin-bottom:12px;">
    API fora do ar
  </div>

  <div id="aba-painel">
    <div style="background:#fff; padding:12px; border-radius:8px; margin-bottom:8px; border:1px solid #dcdcdc;">
      <div style="font-size:12px; color:#555;">Nível da Smart Water & Energy Station</div>
      <div id="val-nivel" style="font-size:20px; font-weight:bold; color:#111;">--</div>
    </div>
    <div style="background:#fff; padding:12px; border-radius:8px; margin-bottom:8px; border:1px solid #dcdcdc;">
      <div style="font-size:12px; color:#555;">Vazão</div>
      <div id="val-vazao" style="font-size:20px; font-weight:bold; color:#111;">--</div>
    </div>
    <div style="background:#fff; padding:12px; border-radius:8px; margin-bottom:12px; border:1px solid #dcdcdc;">
      <div style="font-size:12px; color:#555;">Consumo Elétrico Simulado</div>
      <div id="val-consumo" style="font-size:20px; font-weight:bold; color:#111;">--</div>
    </div>

    <button onclick="carregarDados()" style="width:100%; padding:10px; margin-bottom:6px; cursor:pointer;">Atualizar Dados</button>
    <button onclick="simularLeitura()" style="width:100%; padding:10px; margin-bottom:6px; cursor:pointer;">Registrar Leitura Simulada</button>
    <button onclick="acionarBomba()" style="width:100%; padding:10px; margin-bottom:10px; cursor:pointer;">Acionar Atuador (Bomba Principal)</button>

    <div style="font-size:14px; font-weight:bold; color:#111;">Estado dos Atuadores:</div>
    <div id="lista-atuadores" style="font-size:13px; color:#333; margin-top:4px;">Nenhum registo.</div>
  </div>

  <div id="aba-falhas" style="display:none;">
    <button onclick="carregarDados()" style="width:100%; padding:10px; margin-bottom:10px; cursor:pointer;">Atualizar Alertas de Falha</button>
    <div id="lista-falhas" style="font-size:13px; color:#333;">Nenhuma falha registada.</div>
  </div>
</div>

<script>
const API_URL = "{url_api}/api";
let estadoAtualBomba = "DESLIGADO";

function trocarAba(aba) {{
  document.getElementById("aba-painel").style.display = aba === "painel" ? "block" : "none";
  document.getElementById("aba-falhas").style.display = aba === "falhas" ? "block" : "none";
  carregarDados();
}}

async function carregarDados() {{
  try {{
    const [rLeituras, rFalhas, rAtuadores] = await Promise.all([
      fetch(API_URL + "/leituras"),
      fetch(API_URL + "/falhas"),
      fetch(API_URL + "/atuadores")
    ]);
    if (!rLeituras.ok || !rFalhas.ok || !rAtuadores.ok) throw new Error("API fora do ar");

    const leituras = await rLeituras.json();
    const falhas = await rFalhas.json();
    const atuadores = await rAtuadores.json();

    document.getElementById("banner-erro").style.display = "none";

    if (leituras.length > 0) {{
      const u = leituras[leituras.length - 1];
      document.getElementById("val-nivel").innerText = u.nivelAgua + " %";
      document.getElementById("val-vazao").innerText = u.vazao + " L/s";
      document.getElementById("val-consumo").innerText = u.consumoEletrico + " kWh";
    }}

    if (atuadores.length > 0) {{
      estadoAtualBomba = atuadores[atuadores.length - 1].estado;
      document.getElementById("lista-atuadores").innerHTML = atuadores
        .slice(-3)
        .map(a => `<div>• ${{a.nome}}: <b>${{a.estado}}</b></div>`).join("");
    }}

    if (falhas.length > 0) {{
      document.getElementById("lista-falhas").innerHTML = falhas
        .slice().reverse()
        .map(f => `<div style="background:#fff3cd; padding:8px; border-radius:6px; margin-bottom:6px; border:1px solid #ffeeba;">
          <b style="color:#856404;">Severidade: ${{f.severidade}}</b><br>${{f.descricao}}
        </div>`).join("");
    }} else {{
      document.getElementById("lista-falhas").innerText = "Nenhuma falha registada.";
    }}
  }} catch (e) {{
    document.getElementById("banner-erro").style.display = "block";
  }}
}}

async function simularLeitura() {{
  try {{
    const r = await fetch(API_URL + "/leituras", {{
      method: "POST",
      headers: {{ "Content-Type": "application/json" }},
      body: JSON.stringify({{
        nivelAgua: Number((Math.random() * 100).toFixed(1)),
        vazao: Number((Math.random() * 60).toFixed(1)),
        consumoEletrico: Number((Math.random() * 600).toFixed(1))
      }})
    }});
    if (!r.ok) throw new Error("API fora do ar");
    await carregarDados();
  }} catch (e) {{
    document.getElementById("banner-erro").style.display = "block";
  }}
}}

async function acionarBomba() {{
  try {{
    const novo = estadoAtualBomba === "LIGADO" ? "DESLIGADO" : "LIGADO";
    const r = await fetch(API_URL + "/atuadores", {{
      method: "POST",
      headers: {{ "Content-Type": "application/json" }},
      body: JSON.stringify({{ nome: "Bomba Principal", estado: novo }})
    }});
    if (!r.ok) throw new Error("API fora do ar");
    await carregarDados();
  }} catch (e) {{
    document.getElementById("banner-erro").style.display = "block";
  }}
}}

carregarDados();
</script>
"""
display(HTML(html_app))

### Célula 9: Parar o Backend (Para demonstrar na apresentação o aviso 'API fora do ar' sem travar o App)

In [ ]:
# !pkill -f "spring-boot"
print("O backend continua rodando. O comando pkill foi comentado para não desligar o App.")

O backend continua rodando. O comando pkill foi comentado para não desligar o App.


### Célula 10: Gerar Histórico de Commits da Equipe, Arquivo `.txt` e Pacote `.zip`

In [ ]:
%%bash
git config --global user.name "Rafael Lucca Silva Freitas Bazan"
git config --global user.email "rm561920@fiap.com.br"

cd /content/backend-estacao
git init
git add pom.xml src/main/resources/application.properties
git commit -m "Configuracao inicial do Spring Boot e banco H2 em arquivo"
git add src/main/java/br/com/fiap/estacao/model src/main/java/br/com/fiap/estacao/repository
git commit -m "Implementacao das entidades e repositorios JPA"
git add src/main/java/br/com/fiap/estacao/controller src/main/java/br/com/fiap/estacao/EstacaoApplication.java
git commit -m "Implementacao dos endpoints REST para leituras, atuadores e falhas"
git add README.md
git commit -m "Documentacao dos endpoints e instrucoes de execucao no README"

cd /content/mobile-estacao
git init
git add package.json src/services/api.js
git commit -m "Configuracao do projeto React Native e servico de consumo da API"
git add src/components
git commit -m "Criacao dos componentes IndicadorCard e BannerErro"
git add src/screens App.js
git commit -m "Implementacao das telas de Painel e Falhas com navegacao e tratamento de erro"
git add README.md
git commit -m "Adicao do README com descricao do projeto e integrantes"

cat << 'EOF' > /content/entrega_portal_fiap.txt
INTEGRANTES:
Rafael Lucca Silva Freitas Bazan - RM 561920
Gabriel Assis - RM 562280
Fernando Castanha - RM 562226

LINKS DO GITHUB:
Backend (Spring Boot + H2): https://github.com/SEU_USUARIO/backend-estacao
Mobile (React Native): https://github.com/SEU_USUARIO/mobile-estacao
EOF

cd /content
zip -r -q GS_Smart_Water_Energy_Station.zip backend-estacao mobile-estacao entrega_portal_fiap.txt
echo "Pacote GS_Smart_Water_Energy_Station.zip e entrega_portal_fiap.txt gerados com sucesso!"

Reinitialized existing Git repository in /content/backend-estacao/.git/
On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/
	target/

nothing added to commit but untracked files present (use "git add" to track)
On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/
	target/

nothing added to commit but untracked files present (use "git add" to track)
On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/
	target/

nothing added to commit but untracked files present (use "git add" to track)
On branch master
Untracked files:
  (use "git add <file>..." to include in what will be committed)
	data/
	target/

nothing added to commit but untracked files present (use "git add" to track)
Reinitialized existing Git repository in /content/mobile-estacao/.git/
On branch master
nothing to commit, working tree clean
On branch master
nothing to co